# 用 SAT 求解数独 ― 用 CNF 表示“恰好一个”

## 本章要点

用 SAT（可满足性问题）的求解器来解数独。本章从一个小推理谜题出发，一步步把问题翻译成布尔公式，最后做到数独。

- 什么是 SAT 和 CNF（合取范式）
- 把析取范式（DNF）转换成 CNF 的方法（SymPy 和手写两种）
- “一组里只有一个为 True”这种约束怎么写
- 用 `pysat` 解数独，用 `assumptions` 给出初始盘面

## 可满足性问题（SAT）

**可满足性问题**（SATisfiability problem，SAT）问的是：“是否存在一组变量取值，能让给定的布尔公式为真？”

SAT 是最早被证明为 **NP 完全**的问题，与计算机科学的基础理论、算法、人工智能、硬件设计等许多领域都有关。而且实用的 SAT 求解器非常快，所以“把要解的问题转换成 SAT”这种做法被广泛使用。

本章用 `pysat` 解数独。先用一个小推理谜题，确认布尔公式和 CNF 的写法。

> **推理谜题**
>
> 4 名嫌疑人作了如下供述。
>
> - 甲：“我不是犯人”
> - 乙：“丁是犯人”
> - 丙：“乙是犯人”
> - 丁：“乙想陷害我”
>
> 4 个人里，只有 1 个人说了真话。犯人是谁？

对甲、乙、丙、丁，用布尔变量 **A**、**B**、**C**、**D** 表示“那个人是犯人”。`~A` 表示“甲不是犯人”。

每条供述都写成如下的布尔公式。

| 嫌疑人 | 供述 | 布尔公式 |
|:--|:--|:--|
| 甲 | 我不是犯人 | S1 = ~A |
| 乙 | 丁是犯人 | S2 = D |
| 丙 | 乙是犯人 | S3 = B |
| 丁 | 乙想陷害我 | S4 = ~D |

说真话的只有 1 个人，所以 S1～S4 中**恰好有一个为真**。

先把“恰好一个为真”直着写出来（`&` 是逻辑与，`|` 是逻辑或）。

```
 S1 & ~S2 & ~S3 & ~S4 |
~S1 &  S2 & ~S3 & ~S4 |
~S1 & ~S2 &  S3 & ~S4 |
~S1 & ~S2 & ~S3 &  S4
```

这是“用 OR 连接的 AND 的集合”，称为**析取范式**（DNF，Disjunctive Normal Form）。

SAT 求解器能接受的只有**合取范式**（CNF，Conjunctive Normal Form）。CNF 是“把用 OR 连接的子句，用 AND 排起来”的形式。所以要把“恰好一个为真”改写成另一种形式。

1. 不能有两个以上为真 → `~(Si & Sj)`
2. 不能全部为假 → `~( ~S1 & ~S2 & ~S3 & ~S4 )`

```
~(S1 & S2) & ~(S1 & S3) & ~(S1 & S4) &
~(S2 & S3) & ~(S2 & S4) & ~(S3 & S4) &
~(~S1 & ~S2 & ~S3 & ~S4)
```

用德摩根定律（`~(X & Y) = ~X | ~Y`）变形，就得到 CNF。

**转换结果**（代入 S1 = ~A, S2 = D, S3 = B, S4 = ~D）

```
~S1 | ~S2   →   A  | ~D
~S1 | ~S3   →   A  | ~B
~S1 | ~S4   →   A  |  D
~S2 | ~S3   →  ~D  | ~B
~S2 | ~S4   →  ~D  |  D     ← 恒为真（不约束任何东西）
~S3 | ~S4   →  ~B  |  D
 S1 |  S2 | S3 | S4  →  ~A | D | B | ~D
```

`~D | D` 恒为真，所以这个子句不约束任何东西。它也没有害处，就原样留着了。

## 用 SymPy 从 DNF 变到 CNF

手写转换容易出错，所以交给 SymPy。

In [1]:
from sympy import symbols
from sympy.logic.boolalg import to_cnf

A, B, C, D = symbols("A:D")

# 每人的供述
S1 = ~A
S2 = D
S3 = B
S4 = ~D

# 只有 1 个人说真话（DNF）
dnf = ((S1 & ~S2 & ~S3 & ~S4) |
       (~S1 & S2 & ~S3 & ~S4) |
       (~S1 & ~S2 & S3 & ~S4) |
       (~S1 & ~S2 & ~S3 & S4))

# simplify=True：先化简式子再返回
print("CNF:", to_cnf(dnf, simplify=True))

# simplify=False：按分配律直接展开
cnf_raw = to_cnf(dnf, simplify=False)
print("simplify=False 时会有", len(cnf_raw.args), "个子句")

CNF: A & ~B
simplify=False 时会有 41 个子句


把 `simplify=True` 打开，会先化简式子再返回。结果是 `A & ~B`，也就是“甲是犯人，乙不是犯人”。

用 `simplify=False` 时，会按分配律直接展开，子句数会膨胀到几十个。像 `A | ~A` 这样恒为真的子句也不会被删掉。在数独这种大问题上，这种膨胀会直接变成求解时间。

两种 CNF 得到的答案应该是一样的。

In [2]:
from sympy.logic.inference import satisfiable

print(satisfiable(to_cnf(dnf, simplify=True)))
print(satisfiable(cnf_raw))

{A: True, B: False}
{A: True, B: False, D: False}


`satisfiable()` 返回让式子为真的变量赋值。`A` 为真，所以**甲是犯人**。`C` 没有出现在式子里，所以取哪个值都行（没有约束）。

## 用 pysat 求解

CNF 可以用嵌套列表表示。一个整数对应一个布尔变量，正数表示该变量为真，负数表示假（否定）。比如 `1` 是 A，`-1` 是 ~A。

把上面的 CNF 直接写成数值列表（`A=1, B=2, C=3, D=4`）。

1. 用 `append_formula()` 把 CNF 加进求解器（可以调用多次）
2. 用 `solve()` 求解
3. 用 `get_model()` 取出解（正数为真）

In [3]:
from pysat.solvers import Solver

sat = Solver()
problem = [[1, -4], [1, -2], [1, 4],            # ~S1 | ~S2 等
           [-4, -2], [-4, 4], [-2, 4],          # ~D | D 恒为真
           [-1, 4, 2, -4]]                      # S1 | S2 | S3 | S4

sat.append_formula(problem)
print(sat.solve())
print(sat.get_model())

True
[1, -2, -3, -4]


`[1, -2, -3, -4]` 表示“A 为真，B、C、D 为假”。结论和 SymPy 一样。

## 数独

数独是在 9×9 的格子里填入 1～9 的数字，满足下列条件的谜题。

- 每一行中 1～9 各出现一次
- 每一列中 1～9 各出现一次
- 每个 3×3 区块中 1～9 各出现一次

![数独游戏](images/sudoku01.png)

黑色的数字是题目，灰色的数字是答案。

SAT 的变量只有 `True` / `False` 两个值，所以把“格子 (r, c) 里放数字 n”做成一个布尔变量。这里用三维数组 `bools`。

- 第 0 轴 … 行
- 第 1 轴 … 列
- 第 2 轴 … 数字（1～9）

比如 `bools[4, 1, 3]` 为真，意思是“第 5 行第 2 列放数字 4”。数独从 1 开始计数，但数组下标从 0 开始，请注意区别。

`bools` 里放的是 SAT 的变量编号（编号从 1 开始）。

In [4]:
import numpy as np

# 9×9×9 = 729 个变量（1〜729）
bools = np.arange(1, 9 * 9 * 9 + 1).reshape(9, 9, 9)

数独的约束可以整理成 4 类。

| # | 约束 | 分组方法 |
|---|---|---|
| 1 | 每个格子只放一个数字 | 把数字轴（第 2 轴）作为一组 |
| 2 | 每一行中同一数字只有一个 | 按行和数字分组 |
| 3 | 每一列中同一数字只有一个 | 按列和数字分组 |
| 4 | 每个区块中同一数字只有一个 | 按区块和数字分组 |

它们都是“**某一组里恰好只有一个为 True**”的形式。

## 用 CNF 表示“恰好一个”

变量 `A`、`B`、`C` 中只有一个为真，这个约束可以拆成两个条件。

1. 至少有一个为真: $A | B | C$
2. 不能有两个以上为真: $\neg(A \& B) \land \neg(A \& C) \land \neg(B \& C)$

用德摩根定律，第 2 个条件变成 $ (\neg A | \neg B) \land (\neg A | \neg C) \land (\neg B | \neg C) $。

$$
(A | B | C) \land (\neg A | \neg B) \land (\neg A | \neg C) \land (\neg B | \neg C)
$$

元素有 N 个时，就是 1 个“把全部用 OR 连起来的子句”，加上“对每个二元组写 `~X | ~Y`”的 $N(N-1)/2$ 个。二元组可以用 `itertools.combinations` 生成。

`get_conditions(bools)` 把 `bools` 的**最后一轴**作为一组，返回这个约束。

1. 对最后一轴的元素个数 `n`，生成所有的二元组
2. 为每个组生成用 OR 连起来的子句（至少一个为真）
3. 为每个二元组生成 `~X | ~Y` 子句（两个不能同时为真）

In [5]:
from itertools import combinations


def get_conditions(bools):
    """把最后一轴作为一组，生成“恰好一个为 True”的 CNF"""
    n = bools.shape[-1]
    pairs = np.array(list(combinations(range(n), 2)))

    conditions = []
    # 1. 至少一个为 True
    conditions.extend(bools.reshape(-1, n).tolist())
    # 2. 两个不能同时为 True
    conditions.extend((-bools[..., pairs].reshape(-1, 2)).tolist())
    return conditions

用小例子确认行为。`[[1, 2, 3], [4, 5, 6]]` 表示“有两个组，每组各有 3 个变量”。

- `[1, 2, 3]`, `[4, 5, 6]` … 每个组中至少一个为真
- `[-1, -2]`, `[-1, -3]`, `[-2, -3]` … 1～3 中不能有两个同时为真
- `[-4, -5]`, `[-4, -6]`, `[-5, -6]` … 4～6 也一样

In [6]:
get_conditions(np.array([[1, 2, 3], [4, 5, 6]]))

[[1, 2, 3],
 [4, 5, 6],
 [-1, -2],
 [-1, -3],
 [-2, -3],
 [-4, -5],
 [-4, -6],
 [-5, -6]]

## 生成数独的约束

`get_conditions()` 把最后一轴作为一组，所以行、列、区块的约束要先用 `swapaxes()` **把想分组的轴移到最后**，再传进去。

- 约束 1（每个格子一个）… 原样（最后一轴是数字）
- 约束 2（每行一个）… 交换列和数字
- 约束 3（每列一个）… 交换行和数字

In [7]:
c1 = get_conditions(bools)                     # 每个格子只有一个数字
c2 = get_conditions(np.swapaxes(bools, 1, 2))  # 每行中同一数字只有一个
c3 = get_conditions(np.swapaxes(bools, 0, 2))  # 每列中同一数字只有一个

约束 4（区块）稍微复杂一些。`bools` 是 `(行, 列, 数字)`，所以要把行和列拆成“区块编号”和“区块内的位置”。

1. `reshape(3, 3, 3, 3, 9)` … `(区块行, 行内, 区块列, 列内, 数字)`
2. `swapaxes(1, 2)` … `(区块行, 区块列, 行内, 列内, 数字)`
3. `reshape(9, 9, 9)` … `(区块编号, 区块内的 9 个格子, 数字)`
4. `swapaxes(1, 2)` … `(区块编号, 数字, 区块内的 9 个格子)` ← 分组的轴在最后

这样就得到“每个区块、每个数字，9 个格子里只有一个为真”的约束。

In [8]:
# 按 (区块, 9 个格子, 数字) 的形状整理
blocks = np.swapaxes(bools.reshape(3, 3, 3, 3, 9), 1, 2).reshape(9, 9, 9)
c4 = get_conditions(np.swapaxes(blocks, 1, 2))   # 每个区块中同一数字只有一个

conditions = c1 + c2 + c3 + c4
print(len(conditions), "个子句")

11988 个子句


## 求解空盘面

先在没有提示的状态下求解。只满足数独约束的答案会出来一个（因为没有提示，答案有无穷多个）。

模型是变量编号的列表，所以把它还原成 `(9, 9, 9)`，取出为真的变量的数字（最后一轴的下标 + 1）。

In [9]:
def format_solution(model, shape=(9, 9, 9)):
    """把模型（变量编号的列表）变成 9×9 的数字盘面"""
    grid = np.array(model).reshape(shape)
    return np.where(grid > 0)[2].reshape(9, 9) + 1


sat = Solver()
sat.append_formula(conditions)
print(sat.solve())
format_solution(sat.get_model())

True


array([[2, 4, 6, 1, 3, 5, 8, 7, 9],
       [1, 3, 5, 7, 8, 9, 2, 4, 6],
       [8, 7, 9, 2, 4, 6, 1, 3, 5],
       [5, 6, 3, 4, 1, 2, 7, 9, 8],
       [4, 1, 2, 9, 7, 8, 5, 6, 3],
       [7, 9, 8, 5, 6, 3, 4, 1, 2],
       [6, 5, 4, 3, 2, 1, 9, 8, 7],
       [3, 2, 1, 8, 9, 7, 6, 5, 4],
       [9, 8, 7, 6, 5, 4, 3, 2, 1]])

## 求解实际问题

题目用字符串给出。`0` 是空格子。不是空格子的地方，只要追加“那个数字的变量为真”的约束，就能固定成那个数字。

In [10]:
sudoku_str = """
000000185
007030000
000021400
800000020
003905600
050000004
004860000
000040300
931000000"""

puzzle = np.array([[int(x) for x in line]
                   for line in sudoku_str.strip().split()])

# 对提示格加上“就是那个数字”的约束
r, c = np.where(puzzle != 0)
v = puzzle[r, c] - 1
hints = [[int(x)] for x in bools[r, c, v]]

sat = Solver()
sat.append_formula(conditions + hints)
sat.solve()
solution = format_solution(sat.get_model())
solution

array([[3, 6, 2, 7, 9, 4, 1, 8, 5],
       [4, 1, 7, 5, 3, 8, 2, 6, 9],
       [5, 9, 8, 6, 2, 1, 4, 3, 7],
       [8, 7, 9, 4, 1, 6, 5, 2, 3],
       [2, 4, 3, 9, 7, 5, 6, 1, 8],
       [1, 5, 6, 3, 8, 2, 7, 9, 4],
       [7, 2, 4, 8, 6, 3, 9, 5, 1],
       [6, 8, 5, 1, 4, 9, 3, 7, 2],
       [9, 3, 1, 2, 5, 7, 8, 4, 6]])

`hints` 里的每个子句都只含一个布尔变量（`[x]`）。这表示“x 一定为真”。

## 验算解

确认得到的盘面是否真的满足数独条件。

In [11]:
def check_sudoku(grid):
    """每行、每列、每个区块是否都是 1〜9 各一次"""
    target = set(range(1, 10))
    rows = [set(grid[i, :]) for i in range(9)]
    cols = [set(grid[:, j]) for j in range(9)]
    blocks = [set(grid[i:i+3, j:j+3].ravel())
              for i in range(0, 9, 3) for j in range(0, 9, 3)]
    return all(g == target for g in rows + cols + blocks)


print(check_sudoku(solution))

# 也确认与提示不矛盾
print(bool((puzzle[puzzle > 0] == solution[puzzle > 0]).all()))

True
True


## 用 assumptions 给出提示

除了像 `hints` 那样追加“只含一个变量的子句”，也可以给 `solve()` 的 `assumptions` 传一个想让它为真的变量列表。

`assumptions` 在求解后会被清空，所以**可以复用同一个求解器，接着解别的题目**。要解很多题时这样更方便。

In [12]:
sat = Solver()
sat.append_formula(conditions)                     # 只放入数独的规则

sat.solve(assumptions=bools[r, c, v].tolist())     # 提示作为假设传入
solution2 = format_solution(sat.get_model())
print(check_sudoku(solution2))
solution2

True


array([[3, 6, 2, 7, 9, 4, 1, 8, 5],
       [4, 1, 7, 5, 3, 8, 2, 6, 9],
       [5, 9, 8, 6, 2, 1, 4, 3, 7],
       [8, 7, 9, 4, 1, 6, 5, 2, 3],
       [2, 4, 3, 9, 7, 5, 6, 1, 8],
       [1, 5, 6, 3, 8, 2, 7, 9, 4],
       [7, 2, 4, 8, 6, 3, 9, 5, 1],
       [6, 8, 5, 1, 4, 9, 3, 7, 2],
       [9, 3, 1, 2, 5, 7, 8, 4, 6]])

## 小结

- SAT 是求解“是否存在让布尔公式为真的赋值”的问题。虽然是 NP 完全，但实用的求解器很快
- 求解器只接受 CNF（合取范式）。DNF 不能直接传进去
- “恰好一个为真”用“至少一个”＋“所有二元组都是 `~X | ~Y`”就能写成 CNF
- 数独把“格子 × 数字”做成布尔变量，排列格子、行、列、区块 4 类约束
- 用 `swapaxes()` 把想分组的轴移到最后，就能复用同一个函数
- 初始盘面可以用 `hints`（单变量子句）给，也可以用 `assumptions` 给。要复用时用 `assumptions`